<a href="https://colab.research.google.com/github/michaelc511/langchain-projects/blob/main/4_7_Chi_Wen_Chang_Course_End_Project_1_Crafting_an_AI_Powered_HR_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import the required libraries




In [ ]:
!pip install --force-reinstall langchain-chroma langchain langchain-community langchain-openai chromadb pypdf

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_openai-1.3.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached pypdf-6.13.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached langchain_core-1.4.7-py3-none-any.whl.metadata (4.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.4 MB/s eta 0:00:00
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI, OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
#from langchain.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA
import openai
import os

/tmp/ipykernel_40181/666070815.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


In [ ]:
import importlib.util
import sys

def find_module_path(module_name):
    spec = importlib.util.find_spec(module_name)
    if spec and spec.origin:  # spec.origin is the path to the module file or __init__.py
        return spec.origin
    return f"Module '{module_name}' not found in sys.path or its origin is not available."

def list_package_contents(package_name):
    try:
        package = importlib.import_module(package_name)
        package_path = package.__path__[0] # Get the main path of the package
        print(f"\n--- Contents of {package_name} at {package_path} ---")
        import os
        for root, dirs, files in os.walk(package_path):
            level = root.replace(package_path, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 4 * (level + 1)
            for f in files:
                print(f'{subindent}{f}')
            if level > 2: # Limit depth to avoid excessive output
                del dirs[:] # Don't go deeper
    except ModuleNotFoundError:
        print(f"Package '{package_name}' is not installed or cannot be imported.")
    except Exception as e:
        print(f"Error listing contents for '{package_name}': {e}")

print(f"Python executable: {sys.executable}")
print(f"sys.path: {sys.path}\n")

print(f"langchain module path: {find_module_path('langchain')}")
print(f"langchain.chains module path: {find_module_path('langchain.chains')}")
print(f"langchain_community module path: {find_module_path('langchain_community')}")
print(f"langchain_community.chains module path: {find_module_path('langchain_community.chains')}")

list_package_contents('langchain')
list_package_contents('langchain_community')

Python executable: /usr/bin/python3
sys.path: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']

langchain module path: /usr/local/lib/python3.12/dist-packages/langchain/__init__.py
langchain.chains module path: Module 'langchain.chains' not found in sys.path or its origin is not available.
langchain_community module path: /usr/local/lib/python3.12/dist-packages/langchain_community/__init__.py
langchain_community.chains module path: /usr/local/lib/python3.12/dist-packages/langchain_community/chains/__init__.py

--- Contents of langchain at /usr/local/lib/python3.12/dist-packages/langchain ---
langchain/
    __init__.py
    py.typed
    __pycache__/
        __init__.cpython-312.pyc
    rate_limiters/
        __init__.py
        __pycache__/
            __init__.cpytho

# Loading Documents


In [18]:
from google.colab import drive, userdata

# 1. Mount Drive
#drive.mount('/content/drive')

# 3. Load PDF (Replace with your actual folder path)
# Right-click your Dataset folder in the sidebar -> 'Copy path'
dataset_folder_path = '/content/drive/MyDrive/Dataset'
file_path = os.path.join(dataset_folder_path, 'the_nestle_hr_policy_pdf_2012.pdf')
#print(f"File path: {file_path}")

loader = PyPDFLoader(file_path)
documents = loader.load()


File path: /content/drive/MyDrive/Dataset/the_nestle_hr_policy_pdf_2012.pdf


 # Creating Vector Representation of Texts

In [19]:
from google.colab import userdata
import os

# 1. Setup your credentials first
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. Then run your logic
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings()
vectordb = Chroma.from_documents(texts, embeddings)



# Setting Up Question-Answering System

In [20]:
qa = RetrievalQA.from_chain_type(llm=ChatOpenAI(model_name="gpt-3.5-turbo"), chain_type="stuff", retriever=vectordb.as_retriever())

# Defining Prompt Template

In [21]:
from langchain_core.prompts import PromptTemplate

# Define the prompt template in English
template = """
I am a HR helpful assistant. Please answer the following question in English.
Question: {question}
Answer:
"""

# Create the PromptTemplate instance with the modified English template
prompt = PromptTemplate(
    input_variables=["question"],
    template=template,
)

#  Building Chat Interface with Gradio and Launching the Chat Interface

In [ ]:
import gradio as gr

def add_text(history, text):
    # This ensures content is an empty string, satisfying Pydantic's validation
    history.append({"role": "user", "content": text})
    history.append({"role": "assistant", "content": ""})
    return history, ""

def bot(history):
    # Safety check: ensure history has at least 2 entries
    if not history or len(history) < 2:
        return history

    # Retrieve the user's latest message
    user_message = history[-2]["content"]

    # Run your QA logic
    query = prompt.format(question=user_message)
    # Using invoke() is the standard in modern LangChain
    answer = qa.invoke({"query": query})["result"]

    # Update the assistant's placeholder with the actual answer
    history[-1]["content"] = answer
    return history

    # Run your QA logic
    try:
        query = prompt.format(question=user_message)
        print(f"Formatted Query: {query}")

        answer_raw = qa.invoke({"query": query})
        answer = answer_raw.get("result") if isinstance(answer_raw, dict) else None

        # Ensure answer is always a non-None string for Gradio's Pydantic validation
        if answer is None or not isinstance(answer, str):
            answer = "Sorry, I could not generate a valid answer for that question. Please try rephrasing."

        print(f"Answer received from QA chain: {answer}")

        # Update the assistant's content dictionary, ensuring it exists
        if len(history) > 0 and isinstance(history[-1], dict) and history[-1].get("role") == "assistant":
            history[-1]["content"] = answer
        else:
            # Fallback if the last element is not the expected assistant placeholder
            print(f"Warning: Last history element not as expected for assistant update. Appending new assistant message. Current history: {history}")
            history.append({"role": "assistant", "content": answer})

    except Exception as e:
        error_message = f"An error occurred during QA processing: {e}. Please check the OpenAI API key or the question format."
        print(error_message)
        # Ensure that history[-1]["content"] is always a string even on QA failure
        if len(history) > 0 and isinstance(history[-1], dict) and history[-1].get("role") == "assistant":
            history[-1]["content"] = "Sorry, I encountered an error while processing your request. Please try again or rephrase your question."
        else:
            history.append({"role": "assistant", "content": "Sorry, I encountered an error while processing your request. Please try again or rephrase your question."})

    return history


with gr.Blocks() as demo:
    chatbot = gr.Chatbot([], elem_id="chatbot", height=400, type='messages', allow_tags=False)

    with gr.Row():
        with gr.Column(scale=1):
            txt = gr.Textbox(
                show_label=False,
                placeholder="Enter text and press enter",
                container=False
            )

    txt.submit(add_text, [chatbot, txt], [chatbot, txt]).then(
        bot, chatbot, chatbot
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_40181/4060708096.py:61: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot([], elem_id="chatbot", height=400, type='messages', allow_tags=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cc93fb2828b7d58095.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
